# 04 — FastAPI Model Serving
**MLOps Assignment 01: Heart Disease Prediction**

This notebook checks the project files, installs API dependencies, validates the saved model, tests the API request schema, and starts FastAPI with Uvicorn.

> **Important:** This is an educational machine-learning demonstration, not a medical diagnostic tool. Predictions must not be used for medical decisions.

## Prerequisites
- Open this notebook in the same Colab runtime where the repository is available at `/content/MLOps-Assignment`.
- The repository should contain `api/app.py` and `models/heart_disease_pipeline.joblib`.
- Run cells in order.

## 1. Set the project directory and verify files

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/content/MLOps-Assignment")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}. "
        "Clone or upload your repository to this location first."
    )

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

required_files = [
    Path("api/app.py"),
    Path("api/__init__.py"),
    Path("models/heart_disease_pipeline.joblib"),
    Path("requirements.txt"),
    Path("Dockerfile"),
]
for file_path in required_files:
    print(f"{'OK' if file_path.is_file() else 'MISSING'}: {file_path}")

missing = [str(p) for p in required_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Required project files are missing: " + ", ".join(missing))

## 2. Install dependencies

In [ ]:
%pip install -r requirements.txt
%pip install fastapi uvicorn

## 3. Verify imports and load the trained pipeline

In [ ]:
import fastapi
import uvicorn
import joblib
import pandas as pd

MODEL_PATH = PROJECT_ROOT / "models" / "heart_disease_pipeline.joblib"
model = joblib.load(MODEL_PATH)

print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("Pandas:", pd.__version__)
print("Model type:", type(model).__name__)
print("Pipeline steps:", getattr(model, "named_steps", {}).keys())
print("All required imports and model loading successful.")

## 4. Inspect the API routes

In [ ]:
from api.app import app, FEATURE_COLUMNS, MODEL_PATH as API_MODEL_PATH

print("API title:", app.title)
print("API version:", app.version)
print("API model path:", API_MODEL_PATH)
print("Number of input features:", len(FEATURE_COLUMNS))
print("Feature order:", FEATURE_COLUMNS)
print("\nRegistered routes:")
for route in app.routes:
    methods = ",".join(sorted(getattr(route, "methods", []) or []))
    print(f"{methods:12s} {route.path}")

## 5. Test API validation and prediction in-process

This cell uses FastAPI's `TestClient` to test the API without needing a separate server process. It checks the root and health endpoints, submits a sample payload to `/predict`, and checks that missing fields are rejected.

The sample values are for software testing only; they are **not** a real person's measurements and must not be interpreted medically.

In [ ]:
from fastapi.testclient import TestClient

sample_payload = {
    "age": 54,
    "sex": 1,
    "cp": 0,
    "trestbps": 130,
    "chol": 250,
    "fbs": 0,
    "restecg": 1,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 1,
    "ca": 0,
    "thal": 2,
}

with TestClient(app) as client:
    root_response = client.get("/")
    print("GET / status:", root_response.status_code)
    print("GET / response:", root_response.json())

    health_response = client.get("/health")
    print("\nGET /health status:", health_response.status_code)
    print("GET /health response:", health_response.json())

    prediction_response = client.post("/predict", json=sample_payload)
    print("\nPOST /predict status:", prediction_response.status_code)
    print("POST /predict response:", prediction_response.json())

    invalid_response = client.post("/predict", json={"age": 54})
    print("\nInvalid request status (expected 422):", invalid_response.status_code)
    assert root_response.status_code == 200
    assert health_response.status_code == 200
    assert prediction_response.status_code == 200
    assert invalid_response.status_code == 422
    prediction_result = prediction_response.json()
    assert prediction_result["prediction"] in (0, 1)
    assert 0.0 <= prediction_result["probability_class_1"] <= 1.0

print("\nPASS: API endpoints and input validation work.")

## 6. Start the API server (optional)

Run this cell only if you want to start Uvicorn in the current Colab runtime. It runs in the foreground, so the cell will remain busy until you interrupt it. To avoid that, use the background option in the next cell instead.

**Do not run both server-start cells at the same time.**

In [ ]:
# Foreground server option:
# !uvicorn api.app:app --host 0.0.0.0 --port 8000

## 7. Optional: start Uvicorn in the background and check health

In [ ]:
import subprocess
import time
import urllib.request
import urllib.error

# Stop a previously started server from this notebook, if any.
try:
    if "uvicorn_process" in globals() and uvicorn_process.poll() is None:
        uvicorn_process.terminate()
        uvicorn_process.wait(timeout=5)
except Exception as exc:
    print("Previous server cleanup note:", exc)

uvicorn_log_path = PROJECT_ROOT / "reports" / "uvicorn.log"
uvicorn_log_path.parent.mkdir(parents=True, exist_ok=True)
uvicorn_log = open(uvicorn_log_path, "w", encoding="utf-8")

uvicorn_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "api.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=PROJECT_ROOT,
    stdout=uvicorn_log,
    stderr=subprocess.STDOUT,
)

health_url = "http://127.0.0.1:8000/health"
for attempt in range(20):
    if uvicorn_process.poll() is not None:
        raise RuntimeError(
            "Uvicorn exited early. Check reports/uvicorn.log:\n"
            + uvicorn_log_path.read_text(encoding="utf-8")
        )
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            print("Health HTTP status:", response.status)
            print("Health response:", response.read().decode("utf-8"))
            break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError(
        "API did not become ready. Check reports/uvicorn.log:\n"
        + uvicorn_log_path.read_text(encoding="utf-8")
    )

print("API docs URL (inside this runtime): http://127.0.0.1:8000/docs")
print("Server PID:", uvicorn_process.pid)

## 8. Send a live prediction request

In [ ]:
import json
import urllib.request

request = urllib.request.Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps(sample_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(request, timeout=10) as response:
    print("HTTP status:", response.status)
    print(json.dumps(json.loads(response.read().decode("utf-8")), indent=2))

## 9. View server logs if something fails

In [ ]:
log_file = PROJECT_ROOT / "reports" / "uvicorn.log"
if log_file.exists():
    print(log_file.read_text(encoding="utf-8")[-12000:])
else:
    print("No Uvicorn log found yet. Run the background server cell first.")

## 10. Stop the background server when finished

In [ ]:
if "uvicorn_process" in globals() and uvicorn_process.poll() is None:
    uvicorn_process.terminate()
    try:
        uvicorn_process.wait(timeout=5)
        print("Uvicorn server stopped.")
    except subprocess.TimeoutExpired:
        uvicorn_process.kill()
        print("Uvicorn server force-stopped.")
else:
    print("No running Uvicorn process found.")
try:
    uvicorn_log.close()
except Exception:
    pass

## Completion checklist
- [ ] Project files verified.
- [ ] Dependencies installed.
- [ ] Saved model loads successfully.
- [ ] Root and health endpoints return HTTP 200.
- [ ] Valid prediction request returns HTTP 200.
- [ ] Invalid request returns HTTP 422.
- [ ] Optional live Uvicorn request works.
- [ ] Server stopped when testing is complete.

### Next step
Once the notebook runs successfully, save it in `Notebook/04_FastAPI_Model_Serving.ipynb` in the repository and commit/push it to GitHub. Colab's runtime server is local to the runtime; public access requires a separately configured hosting or tunneling solution.